**văn bản in đậm**# Fine-tune Llama-3.2-1B cho Tóm tắt Tiếng Việt — OpenHust

**Model:** `meta-llama/Llama-3.2-1B-Instruct`

**Dataset:** [`OpenHust/vietnamese-summarization`](https://huggingface.co/datasets/OpenHust/vietnamese-summarization)

**Phương pháp:** QLoRA (4-bit) — Abstractive

##  Khác biệt so với `nam194/vietnews`
| | nam194/vietnews | OpenHust/vietnamese-summarization |
|---|---|---|
| Format | Parquet | **CSV (nhiều file)** |
| Cột | `article`, `abstract` | **`Document`, `Summary`** |
| Word-segment | Có | **Không** (raw text — Llama xử lý tốt hơn) |
| Splits | train/val/test | **Chỉ train** (phải tự split) |
| Loader | `load_dataset()` | **Tải CSV bằng pandas** (HF viewer bị lỗi cast) |

##  Yêu cầu
Cần **HF_TOKEN** cho Llama-3.2 (gated). Request access tại https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct, tạo token, add vào Kaggle Secrets với label `HF_TOKEN`.

## 1. Cài thư viện

In [1]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl evaluate rouge_score sentencepiece huggingface_hub pandas triton

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3

In [2]:
import os, gc, json, warnings
from typing import Dict

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download, login
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    PeftModel, TaskType,
)
from trl import SFTTrainer, SFTConfig
import evaluate
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" {device}")
if device.type == "cuda":
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

 cuda
 GPU: Tesla T4
 VRAM: 15.6 GB


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Login HF

In [4]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in")

Logged in


## 3. Cấu hình

In [21]:
CONFIG = {
    # Model
    "model_name": "meta-llama/Llama-3.2-1B-Instruct",

    # Dataset (Tỷ lệ chia Chuyên gia: Tối đa hóa Train, Mở rộng Test)
    "dataset_repo": "OpenHust/vietnamese-summarization",
    "dataset_file": "bio_medicine.csv",
    "text_column": "Document",
    "summary_column": "Summary",
    "domain_column": "Dataset",
    "filter_domain": None,
    "n_train": 5000,                # Gần như toàn bộ dữ liệu hợp lệ
    "n_val": 100,                   # Vừa đủ để vẽ đường Loss
    "n_test": 500,                  # Đủ lớn để điểm ROUGE cực kỳ uy tín

    # Tokenizer
    "max_length": 512,
    "max_new_tokens": 128,

    # QLoRA (Float16 là bắt buộc cho T4)
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_use_double_quant": True,

    # LoRA (Bộ não 22.5 triệu tham số)
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],

    # Training
    "output_dir": "/content/drive/MyDrive/llama32-openhust-sft", # Lưu vào thư mục mới
    "resume_from_checkpoint": True, # Bật tự động tìm checkpoint
    "num_train_epochs": 1,          # Trở lại 1 Epoch để tránh overfitting trên bộ não lớn
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8, # Effective batch = 16
    "learning_rate": 2e-4,          # Giữ LR ở mức vừa phải (không quá gắt như 3e-4)
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "cosine",
    "max_grad_norm": 0.3,
    "optim": "paged_adamw_8bit",
    "bf16": False,
    "fp16": True,
    "gradient_checkpointing": True,

    # Eval & Save
    "eval_strategy": "steps",
    "eval_steps": 100,
    "save_steps": 100,
    "save_total_limit": 2,
    "logging_steps": 20,

    "seed": 42,
    "max_eval_samples_rouge": 500,  # BẮT BUỘC: Đồng bộ với n_test để chấm điểm toàn bộ 500 bài
}

import os
os.makedirs(CONFIG["output_dir"], exist_ok=True)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

  model_name: meta-llama/Llama-3.2-1B-Instruct
  dataset_repo: OpenHust/vietnamese-summarization
  dataset_file: bio_medicine.csv
  text_column: Document
  summary_column: Summary
  domain_column: Dataset
  filter_domain: None
  n_train: 5000
  n_val: 100
  n_test: 500
  max_length: 512
  max_new_tokens: 128
  load_in_4bit: True
  bnb_4bit_quant_type: nf4
  bnb_4bit_compute_dtype: float16
  bnb_4bit_use_double_quant: True
  lora_r: 32
  lora_alpha: 64
  lora_dropout: 0.05
  lora_target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  output_dir: /content/drive/MyDrive/llama32-openhust-sft
  resume_from_checkpoint: True
  num_train_epochs: 1
  per_device_train_batch_size: 2
  per_device_eval_batch_size: 2
  gradient_accumulation_steps: 8
  learning_rate: 0.0002
  weight_decay: 0.01
  warmup_ratio: 0.05
  lr_scheduler_type: cosine
  max_grad_norm: 0.3
  optim: paged_adamw_8bit
  bf16: False
  fp16: True
  gradient_checkpointing: True
  eval_strateg

## 4. Load Dataset từ CSV

Dataset OpenHust có **dataset viewer bị lỗi cast** trên HuggingFace vì các file CSV có schema không đồng nhất (file `bio_medicine.csv` thừa cột `Unnamed: 0`). Phải tự tải file rồi đọc bằng pandas.

In [8]:
csv_path = hf_hub_download(
    repo_id=CONFIG["dataset_repo"],
    filename=CONFIG["dataset_file"],
    repo_type="dataset",
)
print(f" Downloaded: {csv_path}")
print(f"   Size: {os.path.getsize(csv_path)/1e6:.1f} MB")

df = pd.read_csv(csv_path)
print(f"\n Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")

# Drop cột Unnamed thừa
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Cleaning
before = len(df)
df = df.dropna(subset=[CONFIG["text_column"], CONFIG["summary_column"]])
df = df[df[CONFIG["text_column"]].str.strip().str.len() > 100]
df = df[df[CONFIG["summary_column"]].str.strip().str.len() > 20]
print(f"   After cleaning: {len(df):,} (loại {before - len(df):,})")

if CONFIG["domain_column"] in df.columns:
    print(f"\n   Domain distribution:")
    print(df[CONFIG["domain_column"]].value_counts().to_string())
    if CONFIG["filter_domain"]:
        df = df[df[CONFIG["domain_column"]] == CONFIG["filter_domain"]]
        print(f"\n   Filtered '{CONFIG['filter_domain']}': {len(df):,}")

print(f"\n Sample:")
print(f"   Document: {df[CONFIG['text_column']].iloc[0][:300]}...")
print(f"   Summary:  {df[CONFIG['summary_column']].iloc[0][:200]}")

 Downloaded: /root/.cache/huggingface/hub/datasets--OpenHust--vietnamese-summarization/snapshots/c62ecacab1734ddeda45814650ea62468b63833a/bio_medicine.csv
   Size: 33.6 MB

 Shape: (10652, 3)
   Columns: ['Document', 'Summary', 'Dataset']
   After cleaning: 10,647 (loại 5)

   Domain distribution:
Dataset
vietnews     5657
wikiling     4506
wikimulti     438
ViMs           46

 Sample:
   Document: Đây là một trong những nội dung tại văn bản vừa được UBND TP Hà Nội ban hành về việc tăng cường công tác quản lý nuôi , giết mổ , kinh doanh và sử dụng thịt chó , mèo trên địa bàn .Theo đó , các sở , ngành trên địa bàn phải vào cuộc ngay để hướng tới thay đổi thói quen của người dân khi dùng chó , m...
   Summary:  Các quận , huyện , thị xã tuyên truyền bằng nhiều hình thức để người dân từ bỏ thói quen ăn thịt chó , mèo nhằm tránh nguy cơ mắc bệnh truyền nhiễm ( bệnh dại , xoắn khuẩn , tả ... ) cũng như không gâ


In [9]:
# Shuffle + split train/val/test
df_shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

n_train = min(CONFIG["n_train"], len(df_shuffled) - CONFIG["n_val"] - CONFIG["n_test"])
train_df = df_shuffled.iloc[:n_train]
val_df   = df_shuffled.iloc[n_train : n_train + CONFIG["n_val"]]
test_df  = df_shuffled.iloc[n_train + CONFIG["n_val"] : n_train + CONFIG["n_val"] + CONFIG["n_test"]]

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)

print(f" Splits:")
print(f"   Train: {len(train_ds):,}")
print(f"   Val:   {len(val_ds):,}")
print(f"   Test:  {len(test_ds):,}")

del df, df_shuffled, train_df, val_df, test_df
gc.collect()

 Splits:
   Train: 5,000
   Val:   100
   Test:  500


452

## 5. Format Chat Template

Dataset OpenHust dùng **raw text** (không word-segment) → Llama tokenizer xử lý sạch sẽ. Đây là lợi thế so với vietnews.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

SYSTEM_PROMPT = (
    "Bạn là trợ lý chuyên tóm tắt văn bản tiếng Việt. "
    "Hãy tóm tắt nội dung dưới đây thành 1-3 câu ngắn gọn, "
    "giữ đúng các sự kiện, nhân vật và thông tin chính."
)

def format_for_training(example: Dict) -> Dict:
    article = str(example[CONFIG["text_column"]]).strip()
    summary = str(example[CONFIG["summary_column"]]).strip()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Văn bản:\n{article}\n\nHãy tóm tắt:"},
        {"role": "assistant", "content": summary},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

def format_for_inference(article: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Văn bản:\n{article.strip()}\n\nHãy tóm tắt:"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Test
sample_text = format_for_training(train_ds[0])["text"]
print(" Sample formatted (1500 chars đầu):")
print(sample_text[:1500])
print(f"\n   Tokens: {len(tokenizer.encode(sample_text))}")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

 Sample formatted (1500 chars đầu):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 30 Apr 2026

Bạn là trợ lý chuyên tóm tắt văn bản tiếng Việt. Hãy tóm tắt nội dung dưới đây thành 1-3 câu ngắn gọn, giữ đúng các sự kiện, nhân vật và thông tin chính.<|eot_id|><|start_header_id|>user<|end_header_id|>

Văn bản:
Trước đó , vào khoảng 17h30 ngày 17/8 , tại khu vực chợ phường 7 , TP. Cà Mau , nhiều người dân bất ngờ bị một con chó dữ lao vào tấn công .Sự việc đã làm 7 người bị thương .Tất cả 7 nạn nhân này sau khi bị chó cắn đều đã được vận động đến trung tâm Y tế dự phòng tỉnh để tiến hành lập thủ tục , tiêm ngừa .Những người bị thương chủ yếu là người đi chợ , có địa chỉ tại nhiều phường trên địa bàn TP. Cà Mau .Tiếp nhận thông tin , lực lượng chức năng đã nhanh chóng có mặt tại hiện trường .Tuy nhiên , sau khi cắn nhiều người , con chó này cũng đã bị người dân đập chết .Xác chó được một người đàn ông đem đi bán cho ông L.V . 

In [11]:
train_formatted = train_ds.map(format_for_training, remove_columns=train_ds.column_names)
val_formatted   = val_ds.map(format_for_training, remove_columns=val_ds.column_names)

print(f" Train: {len(train_formatted):,} | Val: {len(val_formatted):,}")

lengths = [len(tokenizer.encode(t)) for t in train_formatted.select(range(min(200, len(train_formatted))))["text"]]
print(f"\n Token length:")
print(f"   Min: {min(lengths)} | Max: {max(lengths)}")
print(f"   Mean: {np.mean(lengths):.0f} | p95: {np.percentile(lengths, 95):.0f}")
print(f"   % vượt {CONFIG['max_length']}: {sum(l > CONFIG['max_length'] for l in lengths)/len(lengths):.1%}")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

 Train: 5,000 | Val: 100

 Token length:
   Min: 252 | Max: 1865
   Mean: 753 | p95: 1370
   % vượt 512: 75.5%


## 6. Load Model QLoRA

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG["load_in_4bit"],
    bnb_4bit_quant_type=CONFIG["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=getattr(torch, CONFIG["bnb_4bit_compute_dtype"]),
    bnb_4bit_use_double_quant=CONFIG["bnb_4bit_use_double_quant"],
)

print(f" Loading {CONFIG['model_name']} (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
model.config.use_cache = False
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=CONFIG["gradient_checkpointing"],
)

if device.type == "cuda":
    print(f" VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

 Loading meta-llama/Llama-3.2-1B-Instruct (4-bit)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

 VRAM: 1.55 GB


In [13]:
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["lora_target_modules"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
if device.type == "cuda":
    print(f" VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.2f} GB")

trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916
 VRAM after LoRA: 1.64 GB


## 7. Training

In [14]:
sft_config = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    max_grad_norm=CONFIG["max_grad_norm"],
    optim=CONFIG["optim"],
    bf16=CONFIG["bf16"],
    gradient_checkpointing=CONFIG["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy=CONFIG["eval_strategy"],
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=CONFIG["logging_steps"],
    report_to="none",
    seed=CONFIG["seed"],
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    processing_class=tokenizer,
)

eff_batch = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]
total_steps = (len(train_formatted) // eff_batch) * CONFIG["num_train_epochs"]
print(f" Trainer ready")
print(f"   Effective batch: {eff_batch}")
print(f"   Estimated steps: ~{total_steps:,}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

 Trainer ready
   Effective batch: 16
   Estimated steps: ~312


In [15]:
if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(" Training starting...")
print("=" * 60)
train_result = trainer.train()
print("=" * 60)
print(f" Done!")
print(f"  Loss: {train_result.training_loss:.4f}")
print(f"  Steps: {train_result.global_step}")
print(f"  Time: {train_result.metrics.get('train_runtime', 0)/60:.1f} phút")
if device.type == "cuda":
    print(f"   Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


 Training starting...


Step,Training Loss,Validation Loss
100,2.129509,2.178754
200,2.073129,2.115744
300,2.050783,2.101035
313,2.050783,2.101034


 Done!
  Loss: 2.1366
  Steps: 313
  Time: 107.2 phút
   Peak VRAM: 6.18 GB


## 8. Lưu Adapter

In [16]:
adapter_path = os.path.join(CONFIG["output_dir"], "final-adapter")
trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)
size_mb = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
) / 1e6
print(f" Adapter: {adapter_path} ({size_mb:.1f} MB)")

 Adapter: /content/drive/MyDrive/llama32-openhust-sft/final-adapter (107.4 MB)


## 9. Inference

In [17]:
model.config.use_cache = True
model.eval()

infer_tokenizer = AutoTokenizer.from_pretrained(adapter_path)
if infer_tokenizer.pad_token is None:
    infer_tokenizer.pad_token = infer_tokenizer.eos_token
infer_tokenizer.padding_side = "left"

@torch.no_grad()
def summarize(article: str, max_new_tokens: int = None, **gen_kwargs) -> str:
    if max_new_tokens is None:
        max_new_tokens = CONFIG["max_new_tokens"]
    prompt = format_for_inference(article)
    inputs = infer_tokenizer(
        prompt, return_tensors="pt", truncation=True,
        max_length=CONFIG["max_length"] - max_new_tokens,
    ).to(device)
    default_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False, num_beams=1, repetition_penalty=1.1,
        eos_token_id=[
            infer_tokenizer.eos_token_id,
            infer_tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        ],
        pad_token_id=infer_tokenizer.pad_token_id,
    )
    default_kwargs.update(gen_kwargs)
    outputs = model.generate(**inputs, **default_kwargs)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return infer_tokenizer.decode(generated, skip_special_tokens=True).strip()

# Demo
test_sample = test_ds[0]
print("DOCUMENT:")
print(f"   {test_sample[CONFIG['text_column']][:400]}...\n")
print("REFERENCE:")
print(f"{test_sample[CONFIG['summary_column']]}\n")
print("GENERATED:")
print(f"{summarize(test_sample[CONFIG['text_column']])}")

DOCUMENT:
   Ổ dịch rubella , được ghi nhận tại Công ty Wanek từ ngày 2 đến 23/1 .Viện Pasteur TP HCM , Sở Y tế , Trung tâm Y tế dự phòng đã điều tra xác minh ổ dịch , lấy mẫu xét nghiệm , cách ly bệnh nhân , triển khai các biện pháp phòng chống .Rubella hay còn gọi là bệnh sởi Đức , ở người bình thường thì không nghiêm trọng , thường không để lại biến chứng .Bệnh lây truyền qua đường hô hấp do ho , hắt hơi , ...

REFERENCE:
Cục Y tế Dự phòng ( Bộ Y tế ) vừa thông báo có 138 trường hợp nghi mắc rubella , trong 29 ca được xét nghiệm chẩn đoán mắc rubella tại một công ty trong khu công nghiệp Mỹ Phước ( thị xã Bến Cát , Bình Dương ) .

GENERATED:


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


, bao gồm : viêm não , suy gan , suy thận , suy tim , suy phổi , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận ,


## 10. Đánh giá ROUGE

In [22]:
rouge = evaluate.load("rouge")
n_eval = min(CONFIG["max_eval_samples_rouge"], len(test_ds))
predictions, references = [], []

for ex in tqdm(test_ds.select(range(n_eval)), desc="Generating"):
    article  = str(ex[CONFIG["text_column"]])
    abstract = str(ex[CONFIG["summary_column"]])
    if not article.strip() or not abstract.strip():
        continue
    pred = summarize(article)
    if pred:
        predictions.append(pred)
        references.append(abstract)

rouge_results = rouge.compute(
    predictions=predictions, references=references, use_stemmer=False,
)
print("=" * 60)
print(f" ROUGE — Llama-3.2-1B + QLoRA / OpenHust ({len(predictions)} bài)")
print("=" * 60)
print(f" ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f" ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f" ROUGE-L: {rouge_results['rougeL']:.4f}")
print("=" * 60)

# Lưu
results = {
    "config": {k: str(v) for k, v in CONFIG.items()},
    "train_loss": float(train_result.training_loss),
    "train_runtime_min": train_result.metrics.get("train_runtime", 0) / 60,
    "rouge": {k: float(v) for k, v in rouge_results.items()},
    "n_eval_samples": len(predictions),
}
with open(os.path.join(CONFIG["output_dir"], "results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(" Saved results.json")

Generating:   0%|          | 0/500 [00:00<?, ?it/s]

 ROUGE — Llama-3.2-1B + QLoRA / OpenHust (499 bài)
 ROUGE-1: 0.4365
 ROUGE-2: 0.1584
 ROUGE-L: 0.2705
 Saved results.json


## 11. Demo nhiều bài

In [24]:
for i in range(5):
    ex = test_ds[i]
    pred = summarize(ex[CONFIG["text_column"]])
    r = rouge.compute(
        predictions=[pred], references=[ex[CONFIG["summary_column"]]], use_stemmer=False,
    )
    print(f"\n{'─'*80}")
    if CONFIG["domain_column"] in ex:
        print(f" #{i} | Domain: {ex[CONFIG['domain_column']]}")
    else:
        print(f" #{i}")
    print(f"{'─'*80}")
    print(f" Doc:  {ex[CONFIG['text_column']][:300]}...")
    print(f"\n Ref:  {ex[CONFIG['summary_column']][:300]}")
    print(f"\n Pred: {pred[:300]}")
    print(f"\n ROUGE-1: {r['rouge1']:.3f} | R-2: {r['rouge2']:.3f} | R-L: {r['rougeL']:.3f}")


────────────────────────────────────────────────────────────────────────────────
 #0 | Domain: vietnews
────────────────────────────────────────────────────────────────────────────────
 Doc:  Ổ dịch rubella , được ghi nhận tại Công ty Wanek từ ngày 2 đến 23/1 .Viện Pasteur TP HCM , Sở Y tế , Trung tâm Y tế dự phòng đã điều tra xác minh ổ dịch , lấy mẫu xét nghiệm , cách ly bệnh nhân , triển khai các biện pháp phòng chống .Rubella hay còn gọi là bệnh sởi Đức , ở người bình thường thì khôn...

 Ref:  Cục Y tế Dự phòng ( Bộ Y tế ) vừa thông báo có 138 trường hợp nghi mắc rubella , trong 29 ca được xét nghiệm chẩn đoán mắc rubella tại một công ty trong khu công nghiệp Mỹ Phước ( thị xã Bến Cát , Bình Dương ) .

 Pred: , bao gồm : viêm não , suy gan , suy thận , suy tim , suy phổi , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , suy thận , s

## 12. Custom Input

In [25]:
# Dataset OpenHust dùng raw text, không cần segment trước
custom = """
Sáng 15/3, tại Hà Nội, Thủ tướng Chính phủ đã chủ trì Hội nghị trực tuyến toàn quốc về
phát triển công nghệ trí tuệ nhân tạo (AI). Thủ tướng nhấn mạnh Việt Nam cần nắm bắt
cơ hội từ cuộc cách mạng AI để phát triển kinh tế xã hội. Theo báo cáo, thị trường AI
tại Việt Nam đang tăng trưởng mạnh với hơn 500 doanh nghiệp hoạt động trong lĩnh vực này.
Bộ Khoa học và Công nghệ đã trình Chiến lược quốc gia về AI đến năm 2030. Nhiều đại biểu
cho rằng cần đầu tư mạnh vào đào tạo nhân lực và thu hút chuyên gia Việt kiều về nước.
"""
print(" Custom Input:")
print(custom)
print("\n Summary:")
print(f" {summarize(custom)}")

 Custom Input:

Sáng 15/3, tại Hà Nội, Thủ tướng Chính phủ đã chủ trì Hội nghị trực tuyến toàn quốc về
phát triển công nghệ trí tuệ nhân tạo (AI). Thủ tướng nhấn mạnh Việt Nam cần nắm bắt
cơ hội từ cuộc cách mạng AI để phát triển kinh tế xã hội. Theo báo cáo, thị trường AI
tại Việt Nam đang tăng trưởng mạnh với hơn 500 doanh nghiệp hoạt động trong lĩnh vực này.
Bộ Khoa học và Công nghệ đã trình Chiến lược quốc gia về AI đến năm 2030. Nhiều đại biểu
cho rằng cần đầu tư mạnh vào đào tạo nhân lực và thu hút chuyên gia Việt kiều về nước.


 Summary:
 Thủ tướng Chính phủ cho biết Việt Nam cần nắm bắt cơ hội từ cuộc cách mạng AI để phát triển kinh tế xã hội.


## 13. Load lại sau

In [ ]:
# # Tham khảo cho session khác:
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# import torch
#
# adapter_path = "/content/drive/MyDrive/llama32-openhust-sft"
# bnb = BitsAndBytesConfig(
#     load_in_4bit=True, bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
# )
# base = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.2-1B-Instruct", quantization_config=bnb,
#     device_map="auto", torch_dtype=torch.bfloat16,
# )
# model = PeftModel.from_pretrained(base, adapter_path)
# model.eval()
# tokenizer = AutoTokenizer.from_pretrained(adapter_path)
# tokenizer.padding_side = "left"
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

---

##  Hạn chế đặc thù của dataset OpenHust

### Ưu điểm so với `nam194/vietnews`

**1. Raw text — Llama xử lý tốt hơn**
Không bị mismatch tokenizer như khi gặp `công_an`, `ma_tuý` ở vietnews. Llama BPE chia subwords tự nhiên hơn → kì vọng ROUGE cao hơn 1-2 điểm.

**2. Multi-domain**
Có cả news và bio_medicine → model học được nhiều phong cách viết, generalization tốt hơn.

**3. Nhỏ gọn hơn**
~50k samples (vs 144k của vietnews) → load nhanh, tiết kiệm disk Kaggle.

### Hạn chế đặc thù dataset OpenHust

**1. Schema không nhất quán**
Một số file CSV có cột `Unnamed: 0`, một số không → dataset viewer của HF bị lỗi cast. Phải tự load CSV bằng pandas, drop cột thừa thủ công. Đây là dấu hiệu dataset chưa được curate kỹ.

**2. Chỉ có split `train`**
Phải tự shuffle + tách train/val/test. Khó so sánh fair với các paper khác dùng cùng dataset (mỗi người split khác nhau).

**3. Trùng lặp giữa các file CSV**
`bio_medicine.csv`, `herding_bio_medicine.csv`, `Kmeans_*.csv` có vẻ là các phiên bản filter/cluster của cùng tập gốc. Phải đọc kỹ tên file, tránh duplicate.

**4. Không có document tả chi tiết**
README ngắn, không nói rõ data đến từ đâu, được crawl thế nào, có filter quality không. Khó đánh giá độ tin cậy của reference summary.

**5. Chất lượng summary biến thiên**
Có summary rất tốt (1-2 câu cô đọng), có summary chỉ là 1 câu trong bài (gần như extractive), có summary thiếu thông tin chính. Inconsistent → model học bị nhiễu.

**6. Tên file gây hiểu nhầm**
File `bio_medicine.csv` thực ra phần lớn vẫn là `vietnews` (xem cột `Dataset`). Nếu muốn thuần medical, phải set `filter_domain="bio_medicine"`.

### Hạn chế chung của approach (giống notebook vietnews)

**7. Llama-3.2 yếu tiếng Việt** — pre-train chủ yếu tiếng Anh, có thể bị lẫn token English. Vistral-7B / PhoGPT sẽ tốt hơn cho tiếng Việt thuần.

**8. Hallucination** — bịa tên, ngày, số. Cần human review cho production.

**9. Max length 1024** — bài >1024 tokens bị truncate, mất thông tin.

**10. ROUGE không công bằng cho abstractive** — summary đúng ý nhưng dùng từ khác sẽ có ROUGE thấp dù chất lượng cao. Nên thêm BERTScore.

### Khi nào nên chọn dataset nào?

**Chọn `nam194/vietnews`:**
- Cần dataset đã pre-process kỹ, có sẵn 3 splits chuẩn
- Domain news thuần
- Cần dataset lớn (144k samples)

**Chọn `OpenHust/vietnamese-summarization`:**
- Muốn raw text (phù hợp Llama, GPT, BARTpho)
- Cần multi-domain (news + medical)
- Train với resource hạn chế (~50k đủ học pattern)

### Kết quả kì vọng

| Method | ROUGE-1 |
|---|---|
| TextRank | ~32-40 |
| Lead-3 | ~38-44 |
| **Llama-3.2-1B + QLoRA / OpenHust** | **~44-52** |
| Llama-3.2-3B + QLoRA | ~48-55 |